# 01b — Write Lance natively (no intermediate files)

**Purpose:** Generate the dataset and write it **straight to Lance fragments**, never landing per-image JPEG files in a Volume. This is one of the two **paved-path native-ingest routes** (`01a` writes the same source to a Delta path-reference table + JPEG files; `01b` writes it straight to Lance). It's the path for customers who control ingestion at the source — e.g. **decoding frames out of videos in bucket storage** and wanting to land the unstructured data on Databricks in the most efficient layout for downstream ML.

Both native routes start from the **same deterministic generation** and diverge only at the terminal format, so the comparison is a clean one-source → two-formats split:

| Path | Notebook | Per-image file I/O | Object-store PUTs |
|---|---|---|---|
| Delta path-ref (native) | `01a` | N JPEG files written | ~N (one per image) |
| **Lance (native, this notebook)** | `01b` | **none** | **~fragments only (dozens)** |
| Lance (convert existing files) | `optional/01_lance_conversion` | N JPEG files read back | ~N GETs, then ~fragments PUTs |

The native path is where Lance's structural write advantage is fully visible: it **never issues the per-image PUT storm** that the Delta path-ref pattern requires. (The optional migration notebook answers a narrower question — *"I already have a curated Volume of JPEGs + a Delta table; is it worth converting?"* — and deliberately pays the file-readback cost.)

Because generation is deterministic (seeded by `(SEED, id)`), the bytes here are **identical** to what `01a` produced — so file/fragment counts are directly comparable. Generation runs behind a `materialize()` barrier before the write is timed (same as `01a`), so `lance_write_s` measures only the fragment write + commit — directly comparable to `01a`'s table-write step. The headline contrast is **file/PUT count**: `n_frag` fragments here vs ~N per-image PUTs on the Delta path.

**Compute:** Databricks Classic Compute — the **same 14-worker cluster as `01a`**, but this notebook pins Ray to **7 nodes** (generation *and* write both run on Ray). The other 7 workers — the ones `01a` gives to Spark for its Delta write — sit idle here, since the Lance path needs no Spark and no SQL Warehouse. Pinning to 7 (not 14) keeps the write node-matched against `01a`'s 7-Spark write.

---

**Output (per size tier):**
- Lance dataset at `/Volumes/{catalog}/{schema}/{volume}/synthetic_lance_{size}/` — the paved Lance dataset `02_training_benchmark.ipynb` trains on
- Metrics JSON at `/Volumes/{catalog}/{schema}/{volume}/artifacts/lance_{size}.json`

**Next:** `02_training_benchmark.ipynb`; full comparison compiled in `03_compile_results.ipynb`.

In [ ]:
# Must install before setup_ray_cluster — installing after restarts the Ray workers.
# ray[data]==2.54.1 pinned: 2.55.0+ added storage_options_provider to lance_datasink,
# a kwarg no released pylance version accepts.
# pyarrow floored not pinned — DBR preinstalls it; an exact pin risks a version conflict.
%pip install -qU "ray[default, data]==2.54.1" pylance numpy pandas "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [0]:
# ── Widgets ─────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 01a_delta_native + 02_training_benchmark.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
lance_subdir  = f"synthetic_lance_{size}"             # paved Lance dataset — 02 trains on this
lance_path    = f"{base_vol}/{lance_subdir}"          # output: native Lance dataset (inline bytes)
artifacts_dir = f"{base_vol}/artifacts"               # write metrics here for notebook 03

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Lance out   : {lance_path}")
print(f"Artifacts   : {artifacts_dir}")
print(f"Categories  : {CATEGORIES}")


In [0]:
import os

# Credentials — set BEFORE setup_ray_cluster so Ray workers inherit them (Ray 2.41+).
os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [0]:
# Ensure the output volume + Ray tmp + artifacts dir exist. No Delta table, no
# image files — this path writes only the Lance dataset.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
for vol_name in [volume, "ray_tmp"]:
    try:
        w.volumes.read(f"{catalog}.{schema}.{vol_name}")
    except Exception:
        w.volumes.create(catalog_name=catalog, schema_name=schema, name=vol_name,
                         volume_type=sdk_catalog.VolumeType.MANAGED)
        print(f"Created volume {catalog}.{schema}.{vol_name}")

ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"
os.makedirs(artifacts_dir, exist_ok=True)
print(f"Ray tmp     : {ray_tmp_path}")
print(f"Artifacts   : {artifacts_dir}")


In [0]:
# Classic Compute Ray cluster.
# Runs on the SAME 14-worker cluster as 01a, but pins Ray to 7 nodes (N_WORKER_NODES=7)
# — NOT all 14 — deliberately. 01a splits its 14 into 7 Ray (generate) + 7 Spark (write);
# to keep the comparison fair, Lance must generate AND write on the same 7-node budget.
# The other 7 workers (the ones 01a hands to Spark) simply sit idle here, since the Lance
# path needs no Spark at all. Bumping Lance to 14 would give it 2x Delta's generation
# nodes — an asymmetry in the other direction.
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

N_WORKER_NODES = 7
CPUS_PER_NODE  = 16

setup_ray_cluster(
    min_worker_nodes=N_WORKER_NODES,
    max_worker_nodes=N_WORKER_NODES,      # fixed size (min == max)
    num_cpus_worker_node=CPUS_PER_NODE,
    num_gpus_worker_node=0,
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

total_cpus = ray.cluster_resources().get("CPU", 0)
print(f"Total CPUs  : {total_cpus:.0f} | alive nodes: {sum(1 for n in ray.nodes() if n['Alive'])}")

In [0]:
import time, os


def dir_stats(path):
    total, nfiles = 0, 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f)); nfiles += 1
            except OSError:
                pass
    return total, nfiles

## Generate synthetic data (once)

`ray.data.range(N).map_batches(generate_batch)` fans generation across the cluster. Each row is seeded by `(SEED, id)`, so generation is deterministic and independent of block partitioning. The image is conditioned on category (hue) so the classification task in `02` is learnable; noise keeps the JPEG in the ~30–300KB range.

In [0]:
import numpy as np


def _make_image(rng, category_idx, n_categories):
    """Procedural RGB image conditioned on category, JPEG-encoded to ~30-300KB."""
    import io
    from PIL import Image

    side = int(rng.integers(256, 512))
    base = np.zeros((side, side, 3), dtype=np.float32)
    hue = category_idx / n_categories
    base[..., 0] = 255 * hue
    base[..., 1] = 255 * (1 - hue)
    base[..., 2] = 128
    noise = rng.integers(0, 60, size=(side, side, 3))
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def generate_batch(batch, seed, categories, embedding_dim):
    ids = batch["id"]
    n_cat = len(categories)
    images, captions, embeddings, cats, brightness, quality = [], [], [], [], [], []
    for _id in ids:
        rng = np.random.default_rng([seed, int(_id)])
        cat_idx = int(rng.integers(0, n_cat))
        images.append(_make_image(rng, cat_idx, n_cat))
        captions.append(f"a photo of a {categories[cat_idx]} " + "x" * int(rng.integers(0, 40)))
        embeddings.append(rng.standard_normal(embedding_dim).astype(np.float32))
        cats.append(categories[cat_idx])
        brightness.append(float(rng.random()))
        quality.append(int(rng.integers(1, 6)))
    return {
        "id":         np.asarray(ids),
        "image":      np.asarray(images, dtype=object),
        "caption":    np.asarray(captions, dtype=object),
        "embedding":  np.asarray(embeddings, dtype=np.float32),
        "category":   np.asarray(cats, dtype=object),
        "brightness": np.asarray(brightness, dtype=np.float32),
        "quality":    np.asarray(quality, dtype=np.int32),
    }

In [0]:
# Generate ONCE and materialize — written straight to Lance, no file-landing step.
override_blocks = max(64, N_ROWS // 5_000)

ds = (
    ray.data.range(N_ROWS, override_num_blocks=override_blocks)
    .map_batches(
        generate_batch,
        fn_kwargs={"seed": SEED, "categories": CATEGORIES, "embedding_dim": EMBEDDING_DIM},
        batch_size=512,
    )
    .materialize()
)
print(f"Generated {ds.count():,} rows")

total_image_bytes = ds.map_batches(
    lambda b: {"nbytes": np.array([sum(len(x) for x in b["image"])])},
    batch_size=512,
).sum("nbytes")
print(f"Raw image bytes: {total_image_bytes / 1e9:.3f} GB")

## Write — Lance (native, inline)

Each Ray write task emits an independent Lance fragment; a single driver-side commit
merges fragment metadata into a new dataset version. Image bytes are written inline,
directly from the generated in-memory blocks — **no per-image file is ever created**,
so the write issues only ~one PUT per fragment instead of one per image.

The write is split into distributed `write_fragments` + an explicit driver-side `commit`
(rather than a one-call high-level writer) for a Databricks-specific reason: Lance's
default commit finalises with a POSIX `rename()`, which the UC Volume FUSE mount does
not implement. Pointing the write at the Volume's underlying `s3://` URI commits with an
S3-native atomic `PutObject` instead; the fragment/commit split is what lets each Ray
task write straight to that `s3://` URI in parallel, with only fragment metadata
returning to the driver.

In [ ]:
import base64, os, pickle, time, lance
import boto3 as _boto3
import pyarrow as pa
import numpy as np
from lance.fragment import write_fragments


# Derive the S3 URI that backs this managed Volume. Lance's local-filesystem commit
# path uses POSIX rename(), unimplemented by the UC FUSE driver; writing to the s3://
# URI uses S3-native atomic PutObject instead. Files land in the same Volume and are
# readable via /Volumes/... after commit.
vol_info      = w.volumes.read(f"{catalog}.{schema}.{volume}")
lance_s3_path = f"{vol_info.storage_location.rstrip('/')}/{lance_subdir}"
_vol_id       = vol_info.volume_id
_aws_region   = _boto3.session.Session().region_name or "us-west-2"


def _s3_storage_options() -> dict:
    """UC credential vending — temporary S3 credentials for the managed volume.

    Databricks workers reach managed storage through UC credential vending, not raw
    EC2 instance-profile IAM. DATABRICKS_HOST/TOKEN are set in env by the credentials
    cell (before setup_ray_cluster so all workers inherit them).
    """
    import requests, os
    resp = requests.post(
        f"{os.environ['DATABRICKS_HOST']}/api/2.1/unity-catalog/temporary-volume-credentials",
        headers={"Authorization": f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
        json={"volume_id": _vol_id, "operation": "WRITE_VOLUME"},
        timeout=10,
    )
    if not resp.ok:
        raise RuntimeError(f"UC credential vending {resp.status_code}: {resp.text}")
    aws = resp.json()["aws_temp_credentials"]
    return {
        "aws_access_key_id":     aws["access_key_id"],
        "aws_secret_access_key": aws["secret_access_key"],
        "aws_session_token":     aws.get("session_token", ""),
        "aws_region":            _aws_region,
    }


def _write_frags(batch: pa.Table) -> dict:
    """Distributed Lance fragment write via write_fragments; driver-side commit later.

    Normalize Ray's Arrow blocks to plain Arrow types Lance accepts:
    image -> large_binary, embedding -> list<float>, strings -> large_string.
    """
    cols = batch.to_pydict()
    normalized = pa.Table.from_pydict(
        {
            "id": pa.array(cols["id"], type=pa.int64()),
            "image": pa.array([bytes(x) for x in cols["image"]], type=pa.large_binary()),
            "caption": pa.array(cols["caption"], type=pa.large_string()),
            "embedding": pa.array(
                [np.asarray(x, dtype=np.float32).tolist() for x in cols["embedding"]],
                type=pa.list_(pa.float32()),
            ),
            "category": pa.array(cols["category"], type=pa.large_string()),
            "brightness": pa.array(cols["brightness"], type=pa.float32()),
            "quality": pa.array(cols["quality"], type=pa.int32()),
        }
    )
    schema = normalized.schema
    fragments = write_fragments(
        normalized.to_reader(), lance_s3_path, schema=schema,
        storage_options=_s3_storage_options(),
    )
    return {
        "fragment_b64": np.asarray([
            base64.b64encode(pickle.dumps(fragment)).decode("ascii") for fragment in fragments
        ], dtype=object),
        "schema_b64": np.asarray([
            base64.b64encode(pickle.dumps(schema)).decode("ascii") for _ in fragments
        ], dtype=object),
    }


# NOTE: generation already ran behind the materialize() barrier in the prior cell, so
# lance_write_s below times only the fragment write + commit (generation excluded) —
# comparable to 01a's table-write step. The headline contrast is still file/PUT count.
t0 = time.time()
fragment_rows = ds.map_batches(_write_frags, batch_format="pyarrow").take_all()
fragments, lance_schema = [], None
for row in fragment_rows:
    fragments.append(pickle.loads(base64.b64decode(row["fragment_b64"])))
    lance_schema = pickle.loads(base64.b64decode(row["schema_b64"]))

op = lance.LanceOperation.Overwrite(lance_schema, fragments)
lance.LanceDataset.commit(lance_s3_path, op, storage_options=_s3_storage_options())
lance_write_s = time.time() - t0

lds = lance.dataset(lance_path)
n_frag = len(lds.get_fragments())
lc_bytes, _ = dir_stats(lance_path)
rows_per_s = N_ROWS / lance_write_s
mb_per_s = (lc_bytes / 1e6) / lance_write_s
print(f"Lance write   : {lance_write_s:6.2f}s | {rows_per_s:>10,.0f} rows/s | {mb_per_s:6.1f} MB/s")
print(f"Lance on-disk : {lc_bytes / 1e9:.3f} GB across {n_frag} fragments")
print(f"Files written : {n_frag} fragments (vs ~{N_ROWS:,} JPEG PUTs on the Delta path)")

## Verify — deterministic round-trip + random access

No Delta table exists in this path, so instead of comparing against files we
**regenerate the probe rows from the same `(SEED, id)`** and confirm the Lance inline
bytes match byte-for-byte. This proves the native write is lossless and that the bytes
are identical to what `01a` produced (same seed → same JPEG). Lance point-lookup latency
is timed too — roughly constant regardless of row position (O(1) fragment addressing).

In [0]:
probe_ids = [0, N_ROWS // 2, N_ROWS - 1]

# Lance take([n]) addresses by ROW INDEX, not the id column. Use to_table(filter=...)
# to look up by id for the round-trip comparison; the raw take() below times O(1) access.
print("Lance random-access latency (row-index addressing):")
for idx in probe_ids:
    t0 = time.time()
    lds.take([idx])
    print(f"  take index={idx:>12,}: {(time.time() - t0) * 1000:6.2f} ms")

print("\nLance id-value lookup (for round-trip comparison):")
lance_rows = {}
for pid in probe_ids:
    t0 = time.time()
    tbl = lds.to_table(filter=f"id = {pid}", columns=["id", "image"])
    lance_rows[pid] = tbl.to_pylist()[0]["image"]
    print(f"  scan id={pid:>12,}: {(time.time() - t0) * 1000:6.2f} ms")

# Regenerate each probe row from the same seed — deterministic, so bytes must match.
def _regen_image(pid):
    rng = np.random.default_rng([SEED, int(pid)])
    cat_idx = int(rng.integers(0, len(CATEGORIES)))
    return _make_image(rng, cat_idx, len(CATEGORIES))

roundtrip_ok = True
print("\nRound-trip (regenerated bytes == Lance inline):")
for pid in probe_ids:
    regen = _regen_image(pid)
    stored = bytes(lance_rows.get(pid))
    ok = regen == stored
    roundtrip_ok = roundtrip_ok and ok
    print(f"  id={pid:>12,}: {'OK' if ok else 'MISMATCH':>8}  ({len(stored) / 1024:.0f} KB)")
print(f"\nround-trip all OK: {roundtrip_ok}")

## ETL benchmark — backfill a new column (distributed)

Compute a derived column once and add it to the existing dataset. Derived column: the L2 norm of the embedding — a stand-in for any UDF-computed feature.

**Distributed on Ray, to match Delta's distributed Spark backfill.** 01a's backfill is a Spark `UPDATE` that runs across its 7 Spark workers, so a fair wall-clock comparison must run Lance's backfill across the cluster too — not single-threaded on the driver (`lds.add_columns(...)`), which was the earlier apples-to-oranges bug. We fan one `merge_columns` task per fragment across Lance's 7 Ray workers (mirroring the write path in the cell above), each writing **only the new column file** for its fragment, then a single driver-side `Merge` commit stitches the updated fragments into a new dataset version.

Two axes to read separately in `03`:
- **Bytes written** — Lance writes only the new column file per fragment; Delta (inline) rewrites whole row groups, dragging the image bytes along. This is Lance's structural, unambiguous win.
- **Wall-clock** — now a compute-vs-compute comparison, node-matched at 7 Ray workers vs 7 Spark workers, no longer skewed by Lance running single-threaded. (Note: 01a keeps Ray up during its backfill so its Spark `UPDATE` also runs on a 7-node budget, not the full 14.)

Bytes written is measured as the files present *after* the merge but not *before* (always ≥ 0), rather than differencing two whole-directory totals — the latter can go negative when Lance version cleanup or eventual-consistent FUSE listing drops superseded fragment files.

In [0]:
import pyarrow as pa
import pickle


def compute_norm(record_batch):
    """BatchUDF: receives a pyarrow.RecordBatch, returns the new column."""
    embs = np.stack(record_batch.column("embedding").to_pylist()).astype("float32")
    norms = np.linalg.norm(embs, axis=1).astype("float32")
    return pa.record_batch({"embedding_norm": pa.array(norms)})


def dir_file_sizes(path):
    """path -> {file: size}. Lets us measure exactly the files a backfill ADDS,
    instead of subtracting two whole-directory totals (which goes negative when
    version cleanup / eventual-consistent FUSE listing drops old fragment files)."""
    sizes = {}
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try:
                sizes[fp] = os.path.getsize(fp)
            except OSError:
                pass
    return sizes


# ── Lance: DISTRIBUTED add-column backfill across Ray ──────────────────────
# Mirror of the write path (cell 11): one Ray task per fragment merges the new column
# into that fragment (reads only `embedding`, writes only the new column file — existing
# data files are untouched), then a single driver-side Merge commit stitches the updated
# fragments into a new dataset version. 01a's Delta backfill is a Spark UPDATE distributed
# across the same cluster, so this keeps the wall-clock comparison compute-vs-compute
# (Ray workers vs Spark executors), not the earlier single-threaded-driver vs Spark.
lds_rw = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())
# Idempotent: drop the column if a previous run left it, so we measure the full cost.
if "embedding_norm" in lds_rw.schema.names:
    lds_rw.drop_columns(["embedding_norm"])
    lds_rw = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())

read_version = lds_rw.version
frag_ids     = [f.fragment_id for f in lds_rw.get_fragments()]
before       = dir_file_sizes(lance_path)


@ray.remote(num_cpus=1)
def _merge_norm_into_fragment(frag_id, dataset_uri):
    """Runs on a Ray worker: merge the derived column into one fragment.

    Reopens the dataset with fresh vended credentials (same pattern as the write path's
    per-task _s3_storage_options call), then merge_columns writes ONLY the new column
    file for this fragment and returns its updated metadata for the driver-side commit.
    """
    import lance, pickle
    ds = lance.dataset(dataset_uri, storage_options=_s3_storage_options())
    frag = ds.get_fragment(frag_id)
    new_frag_meta, new_schema = frag.merge_columns(compute_norm, columns=["embedding"])
    return pickle.dumps((new_frag_meta, new_schema))


t0 = time.time()
results       = ray.get([_merge_norm_into_fragment.remote(fid, lance_s3_path) for fid in frag_ids])
merged        = [pickle.loads(r) for r in results]
new_fragments = [m[0] for m in merged]
new_schema    = merged[0][1]

op = lance.LanceOperation.Merge(new_fragments, new_schema)
lance.LanceDataset.commit(lance_s3_path, op, read_version=read_version,
                          storage_options=_s3_storage_options())
lance_backfill_s = time.time() - t0

after             = dir_file_sizes(lance_path)
etl_bytes_written = sum(sz for fp, sz in after.items() if fp not in before)  # new files only, always >= 0
lc_bytes_after    = sum(after.values())
print(f"Lance backfill : {lance_backfill_s:6.2f}s | +{etl_bytes_written / 1e6:,.1f} MB new-column bytes "
      f"| {len(frag_ids)} fragments merged in parallel")

In [0]:
import json

# ── common block ── identical key names across 01a / 01b (and optional 01) so 03 can
# stack the artifacts into one table with no per-format key mapping. Format-specific
# detail is kept below in `raw`.
# Generation runs behind a materialize barrier (prior cell), so it is NOT in
# lance_write_s — target_write_s == write_total_s here: there is no intermediate
# file-landing or readback step, which is the whole point of the native path.
common = {
    "path_label":        "lance_native",              # 01b = write straight to Lance, no file landing
    "write_total_s":     round(lance_write_s, 3),
    "target_write_s":    round(lance_write_s, 3),
    "n_output_files":    int(n_frag),                 # fragments — vs 01a's ~N per-image PUTs
    "on_disk_bytes":     int(lc_bytes),
    "etl_backfill_s":    round(lance_backfill_s, 3),
    "etl_bytes_written": int(etl_bytes_written),      # new column files only (measured, always >= 0)
    "roundtrip_ok":      bool(roundtrip_ok),
}

lance_native_metrics = {
    "size":   size,
    "n_rows": int(N_ROWS),
    "common": common,
    "raw": {                                          # format-specific detail
        "raw_image_gb":     round(total_image_bytes / 1e9, 4),
        "lance_write_s":    round(lance_write_s, 3),
        "lc_bytes":         int(lc_bytes),
        "n_frag":           int(n_frag),
        "lance_backfill_s": round(lance_backfill_s, 3),
        "lc_bytes_after":   int(lc_bytes_after),
        "roundtrip_ok":     bool(roundtrip_ok),
    },
}
out_path = f"{artifacts_dir}/lance_{size}.json"
with open(out_path, "w") as f:
    json.dump(lance_native_metrics, f, indent=2)
print(f"Wrote {out_path}")
print(json.dumps(lance_native_metrics, indent=2))

## Done — native Lance artifact ready

The Lance dataset was written directly from generation, with no per-image file landing
step — `n_frag` fragments instead of ~N JPEG PUTs. It lands at `synthetic_lance_{size}`,
the paved dataset `02_training_benchmark.ipynb` trains on. Metrics are in
`artifacts/lance_{size}.json`.

**Compare in `03_compile_results.ipynb`:**
- `01b` (Lance native) vs `01a` (Delta path-ref native) on **file/PUT count** — the small-file story, the core paved-path comparison.
- (Optional) `01b` vs `optional/01_lance_conversion` — the cost the read-back-and-inline migration pass adds.